## 3.0 LLMs VS Chat Models
### Predict Text with LLMs

In [ ]:
from langchain.llms.openai import OpenAI
from langchain.chat_models import ChatOpenAI

llm = OpenAI()
chat = ChatOpenAI()

a = llm.predict("What time is best to sleep?")
b = chat.predict("What time is best to sleep?")

## 3.1 Predict Messages 
### w. Chat Model

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.schema import SystemMessage, AIMessage, HumanMessage # message constructors

chat = ChatOpenAI(temperature=0.1)

messages = [
    SystemMessage(content="You are a geography expert. And you only reply in Italian."), # for context setting
    AIMessage(content="Ciao, mi chiamo Paolo!"),
    HumanMessage(content="What is the distance between Mexico and Thailand. Also, what is your name?"),
]

chat.predict_messages(messages)

## 3.2 Prompt Templates
1. PromptTemplate creates template using string. It's for predicting text.
2. ChatPromptTemplate creates template from message.

In [ ]:
# 1. PromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate

chat = ChatOpenAI(temperature=0.1)

template = PromptTemplate.from_template(
    "What is the distance between {country_a} and {country_b}"
)
prompt = template.format(country_a="Mexico", country_b="Thailand")

chat.predict(prompt)

In [ ]:
# 2. ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

chat = ChatOpenAI(temperature=0.1)

template = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert. And you only reply in {language}."),
    ("ai", "Ciao, mi chiamo {name}!"),
    ("human", "What is the distance between {country_a} and {country_b}. Also, what is your name?"),
])

prompt = template.format_messages(
    language="Greek",
    name="Socrates",
    country_a="Mexico",
    country_b="Thailand",
)

chat.predict_messages(prompt)

## 3.3 OutputParser and LCEL
- OutputParser
    - transform the response of LLM
    - 예제는 Str으로 출력된 Response를 List로 변환하는 Parser
- LCEL(LangChain Expression Language)
    - combine templates, LLM calls, responses
    - only need Chat model, template, OutputParser

In [ ]:
# OutputParser Example 1.
from langchain.schema import BaseOutputParser

class CommaOutputParser(BaseOutputParser):
    def parse(self, text):
        return text.strip().split(",") # 쉼표를 기준으로 string을 잘라 array로 만드는 OutputParser
    
p = CommaOutputParser()
p.parse("Hello,how,are,you")

# result = ['Hello', 'how', 'are', 'you']

In [ ]:
# OutputParser Example 2. 
from langchain.schema import BaseOutputParser

class CommaOutputParser(BaseOutputParser):
    def parse(self, text):
        items = text.strip().split(",")
        return list(map(str.strip, items)) # strip on each one of item in the items list.
    
p = CommaOutputParser()
p.parse("Hello, how, are, you")

# result = ['Hello', 'how', 'are', 'you']

In [ ]:
# Using the OutputParser
from langchain.prompts import ChatPromptTemplate
from langchain.schema import BaseOutputParser

template = ChatPromptTemplate.from_messages([
    ("system", "You are a list generating machine. Everything you are asked will be answered with a comma separated list of max {max_items} in lowercase. Do NOT reply with anything else."),
    ("human", "{question}"),
])

prompt = template.format_messages(
    max_items=10,
    question="What are the colours?",
)

result = chat.predict_messages(prompt)
# result = AIMessage(content='red, orange, ...')

class CommaOutputParser(BaseOutputParser):
    def parse(self, text):
        items = text.strip().split(",")
        return list(map(str.strip, items)) # strip on each one of item in the items list.
    
p = CommaOutputParser()
p.parse(result.content)

# result = ['red', 'orange', ...]

In [ ]:
# Chain with LCEL
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema import BaseOutputParser

chat = ChatOpenAI(temperature=0.1)

template = ChatPromptTemplate.from_messages([
    ("system", "You are a list generating machine. Everything you are asked will be answered with a comma separated list of max {max_items} in lowercase. Do NOT reply with anything else."),
    ("human", "{question}"),
])

class CommaOutputParser(BaseOutputParser):
    def parse(self, text):
        items = text.strip().split(",")
        return list(map(str.strip, items))

# LangChain does: template.format messages | chat.predict | parser.parse
chain = template | chat | CommaOutputParser()
chain.invoke({
    "max_items": 5,
    "question":"What are the pokemons?",
})

## 3.4 Chaining Chains
### LCEL working logic

- [Components those chains could have](https://python.langchain.com/docs/concepts/runnables/):
| Component | Input Type | Output Type |
| --- | --- | --- |
| Prompt | dictionary | PromptValue |
| ChatModel | a string, list of chat messages or a PromptValue | ChatMessage |
| LLM | a string, list of chat messages or a PromptValue | String |
| OutputParser | the output of an LLM or ChatModel | Depends on the parser |
| Retriever | a string | List of Documents |
| Tool | a string or dictionary, depending on the tool | Depends on the tool |

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    temperature=0.1,
    # 실시간 응답
    streaming=True, 
    callbacks=[StreamingStdOutCallbackHandler()],
)

chef_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a world-class international chef. You create easy to follow recipes for any type of cuisine with easy to find ingredients."),
    ("human","I want to cook {country} cuisine.")
])

chef_chain = chef_prompt | chat

veg_chef_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a vegetarian chef specialized on making traditional recipies vegetarian. You find alternative ingredients and explain their preparation. You don't radically modify the recipe. If there is no alternative for a food just say you don't know how to replace it."),
    ("human","{recipe}"),
])

veg_chain = veg_chef_prompt | chat

final_chain = {"recipe":chef_chain} | veg_chain
final_chain.invoke(
    {"country":"Indian"},
)